# Logistic Regression vs Random Forest

In [7]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report
from pathlib import Path
import sys

# Adjust path to project root to import config
sys.path.append("..")
from mlops.config import PROCESSED_DATA_DIR

In [8]:
# Setup MLflow
MLFLOW_TRACKING_URI = "http://localhost:5000"
EXPERIMENT_NAME = "credit-fraud-detection"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='/tmp/mlflow/artifacts/1', creation_time=1771963347830, experiment_id='1', last_update_time=1771963347830, lifecycle_stage='active', name='credit-fraud-detection', tags={}>

In [9]:
# Load Data
train_path = PROCESSED_DATA_DIR / "train.csv"
test_path = PROCESSED_DATA_DIR / "test.csv"

print(f"Loading data from {train_path}...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target_col = "Class"
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Loading data from /Users/serkona/Desktop/itmo/2/mlops/data/processed/train.csv...
Train shape: (227845, 30)
Test shape: (56962, 30)


In [10]:
def train_and_log(model, run_name, params=None):
    """
    Trains a model, logs parameters and metrics to MLflow.
    """
    with mlflow.start_run(run_name=run_name):
        # Log parameters
        if params:
            mlflow.log_params(params)
        
        # Train model
        print(f"Training {run_name}...")
        model.fit(X_train, y_train)
        
        # Predictions
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        # Metrics
        recall = recall_score(y_test, y_pred, pos_label=1)
        f1 = f1_score(y_test, y_pred)
        acc = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob)
        

        print(f"  Recall:   {recall:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Accuracy: {acc:.4f}")
        print(f"  ROC AUC:  {roc_auc:.4f}")
        
        
        # Log metrics
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", roc_auc)
        
        
        # Log model
        mlflow.sklearn.log_model(model, "model")
        print(f"Run '{run_name}' completed and logged to MLflow.")

### Experiment 1: Logistic Regression

In [11]:
# Logistic Regression with default parameters
lr_params = {"C": 1.0, "max_iter": 1000, "random_state": 42}
lr_model = LogisticRegression(**lr_params)

train_and_log(lr_model, "LogisticRegression_Default", lr_params)

Training LogisticRegression_Default...


/Users/serkona/anaconda3/envs/mlops/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Recall:   0.7143
  F1 Score: 0.7692
  Accuracy: 0.9993
  ROC AUC:  0.9518


2026/02/24 23:36:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/02/24 23:36:50 INFO mlflow.tracking._tracking_service.client: 🏃 View run LogisticRegression_Default at: http://localhost:5000/#/experiments/1/runs/ce88aef63e7243279e9b3604a6fffc4c.
2026/02/24 23:36:50 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.


Run 'LogisticRegression_Default' completed and logged to MLflow.


In [12]:
# Logistic Regression with regularization (C=0.1)
lr_params_2 = {"C": 0.1, "max_iter": 1000, "random_state": 42}
lr_model_2 = LogisticRegression(**lr_params_2)

train_and_log(lr_model_2, "LogisticRegression_C_0.1", lr_params_2)

Training LogisticRegression_C_0.1...


/Users/serkona/anaconda3/envs/mlops/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Recall:   0.6735
  F1 Score: 0.7416
  Accuracy: 0.9992
  ROC AUC:  0.9538


2026/02/24 23:36:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/02/24 23:36:59 INFO mlflow.tracking._tracking_service.client: 🏃 View run LogisticRegression_C_0.1 at: http://localhost:5000/#/experiments/1/runs/4e5213c2be594f1a9609d4e7dcafc31b.
2026/02/24 23:36:59 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.


Run 'LogisticRegression_C_0.1' completed and logged to MLflow.


### Experiment 2: Random Forest

In [13]:
# Random Forest (baseline)
rf_params = {"n_estimators": 50, "max_depth": 5, "random_state": 42}
rf_model = RandomForestClassifier(**rf_params)

train_and_log(rf_model, "RandomForest_Baseline", rf_params)

Training RandomForest_Baseline...
  Recall:   0.7449
  F1 Score: 0.8202
  Accuracy: 0.9994
  ROC AUC:  0.9624


2026/02/24 23:37:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/02/24 23:37:48 INFO mlflow.tracking._tracking_service.client: 🏃 View run RandomForest_Baseline at: http://localhost:5000/#/experiments/1/runs/2b18b63a5bf64b12b4ebb17bc59534c8.
2026/02/24 23:37:48 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.


Run 'RandomForest_Baseline' completed and logged to MLflow.


In [14]:
# Random Forest (deeper)
rf_params_2 = {"n_estimators": 100, "max_depth": 15, "random_state": 42}
rf_model_2 = RandomForestClassifier(**rf_params_2)

train_and_log(rf_model_2, "RandomForest_Deep", rf_params_2)

Training RandomForest_Deep...
  Recall:   0.8061
  F1 Score: 0.8634
  Accuracy: 0.9996
  ROC AUC:  0.9813


2026/02/24 23:39:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/02/24 23:39:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run RandomForest_Deep at: http://localhost:5000/#/experiments/1/runs/adead19ebf0a416c9c175564a9794172.
2026/02/24 23:39:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.


Run 'RandomForest_Deep' completed and logged to MLflow.


### Comparison

In [18]:
# Retrieve runs and compare metrics
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
if not runs.empty:
    runs = runs.sort_values("metrics.roc_auc", ascending=False)
    cols = ["tags.mlflow.runName", "metrics.recall", "metrics.f1_score", "metrics.accuracy", "metrics.roc_auc"]
    print("Top models by ROC AUC:")
    print(runs[cols].head())
else:
    print("No runs found.")

Top models by ROC AUC:
          tags.mlflow.runName  metrics.recall  metrics.f1_score  \
0           RandomForest_Deep        0.806122          0.863388   
1       RandomForest_Baseline        0.744898          0.820225   
2    LogisticRegression_C_0.1        0.673469          0.741573   
3  LogisticRegression_Default        0.714286          0.769231   

   metrics.accuracy  metrics.roc_auc  
0          0.999561         0.981260  
1          0.999438         0.962431  
2          0.999192         0.953750  
3          0.999263         0.951772  
